<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/ESAA_YB_3%EC%A1%B0_summer_project_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3. 데이터 병합 및 모델링 구상

In [1]:
!pip install catboost optuna shap -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as plt_sns
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import optuna
import shap
import warnings

# Optuna 및 기본 경고 숨김
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. 데이터 로드 및 전처리
df_europe = pd.read_csv('/content/drive/MyDrive/YB_data/Summer_Project/europe_climate.csv')
df_europe['date'] = pd.to_datetime(df_europe['date'])
df_europe['year'] = df_europe['date'].dt.year
df_europe['month'] = df_europe['date'].dt.month
monthly_europe = df_europe.groupby(['year', 'month'])[['mean_t2m_celsius', 'mean_z500', 'mean_slp_hpa']].mean().reset_index()

df_area = pd.read_csv('/content/drive/MyDrive/YB_data/Summer_Project/area_long.csv')[['year', 'month_num', 'area_value']].rename(columns={'month_num': 'month'})
df_ao = pd.read_csv('/content/drive/MyDrive/YB_data/Summer_Project/AO_index.csv')[['YEAR', 'MONTH', 'AO_Index']].rename(columns={'YEAR': 'year', 'MONTH': 'month'})

df_arctic = pd.read_csv('/content/drive/MyDrive/YB_data/Summer_Project/Arctic_SST_Analysis_Data.csv')
df_arctic['time'] = pd.to_datetime(df_arctic['time'])
df_arctic['year'] = df_arctic['time'].dt.year
df_arctic['month'] = df_arctic['time'].dt.month
df_arctic = df_arctic.drop(columns=['time'])
df_arctic = df_arctic.groupby(['year', 'month']).mean().reset_index()

# 2. 데이터 병합
df = monthly_europe.merge(df_area, on=['year', 'month'], how='inner')
df = df.merge(df_ao, on=['year', 'month'], how='inner')
df = df.merge(df_arctic, on=['year', 'month'], how='inner')
df = df.sort_values(['year', 'month']).reset_index(drop=True)

# 3. 피처 엔지니어링 & 데이터 누수(Leakage) 방지
# 실제 예측 시 사용할 수 있는 과거 정보만 활용하기 위해 Lag 변수 생성
features_to_lag = [
    'area_value', 'AO_Index', 'FRSEAICE', 'TSKINWTR',
    'T10M', 'HFLUXWTR', 'EFLUXWTR'
]

for col in features_to_lag:
    df[f'{col}_lag1'] = df[col].shift(1)
    df[f'{col}_lag3'] = df[col].shift(3)
    df[f'{col}_lag6'] = df[col].shift(6)
    df[f'{col}_roll3_mean'] = df[col].shift(1).rolling(window=3).mean()

# 해빙 변동성(3·6·12개월 표준편차) 생성
df['FRSEAICE_roll3_std'] = df['FRSEAICE'].shift(1).rolling(3).std()
df['FRSEAICE_roll6_std'] = df['FRSEAICE'].shift(1).rolling(6).std()
df['FRSEAICE_roll12_std'] = df['FRSEAICE'].shift(1).rolling(12).std()

df['area_value_roll3_std'] = df['area_value'].shift(1).rolling(3).std()
df['area_value_roll6_std'] = df['area_value'].shift(1).rolling(6).std()
df['area_value_roll12_std'] = df['area_value'].shift(1).rolling(12).std()

# 계절성을 표현하기 위한 월 주기 변수 생성
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df = df.dropna().reset_index(drop=True)

TARGET = 'mean_t2m_celsius'

# 현재 월 정보는 미래 예측 시 사용할 수 없으므로 제외
drop_cols = ['month', TARGET, 'mean_z500', 'mean_slp_hpa'] + features_to_lag

train_df = df[df['year'] <= 2023].reset_index(drop=True)
test_df = df[(df['year'] >= 2024) & (df['year'] <= 2025)].reset_index(drop=True)

# 4. Ablation Study (TimeSeriesSplit)
print("4. Ablation Study (TimeSeriesSplit)")

# 해빙 변수 포함 여부에 따른 성능 비교를 위해 세 가지 피처셋 구성
all_features = [c for c in df.columns if c not in drop_cols]
sea_ice_cols = [c for c in all_features if 'area' in c or 'FRSEAICE' in c]
sea_ice_std_cols = [c for c in sea_ice_cols if 'std' in c]

features_m1 = [c for c in all_features if c not in sea_ice_cols]  # 해빙 변수 제외
features_m2 = [c for c in all_features if c not in sea_ice_std_cols] # 해빙 변수 포함
features_m3 = all_features     # 해빙 + 변동성 변수 포함

# 시계열 교차검증(TimeSeriesSplit) 설정
tscv = TimeSeriesSplit(n_splits=5)

def evaluate_tscv(features_list):
    cv_rmses = []
    X, y = train_df[features_list], train_df[TARGET]

    for train_idx, val_idx in tscv.split(X):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        model = lgb.LGBMRegressor(n_estimators=200, random_state=42, verbose=-1)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        cv_rmses.append(np.sqrt(mean_squared_error(y_val, preds)))

    return np.mean(cv_rmses)

rmse_m1 = evaluate_tscv(features_m1)
rmse_m2 = evaluate_tscv(features_m2)
rmse_m3 = evaluate_tscv(features_m3)

# Ablation Study 결과 출력
print(f"{'Model':<20} | {'RMSE':<8} | {'ΔRMSE':<8} | {'Improvement':<12}")
print("-" * 55)
print(f"{'No Sea Ice':<20} | {rmse_m1:.4f}   | {'-':<8} | {'-':<12}")

diff_2 = rmse_m2 - rmse_m1
imp_2 = (rmse_m1 - rmse_m2) / rmse_m1 * 100
print(f"{'+ Sea Ice':<20} | {rmse_m2:.4f}   | {diff_2:+.4f}  | {imp_2:+.2f}%")

diff_3 = rmse_m3 - rmse_m2
imp_3 = (rmse_m2 - rmse_m3) / rmse_m2 * 100
print(f"{'+ Sea Ice Std':<20} | {rmse_m3:.4f}   | {diff_3:+.4f}  | {imp_3:+.2f}%")
print("-" * 55)

# 성능이 가장 우수한 피처셋을 최종 모델에 사용
best_rmse = min(rmse_m1, rmse_m2, rmse_m3)
if best_rmse == rmse_m1:
    print("결론 : 해빙 변수를 제외한 모델(No Sea Ice)이 가장 우수하여 이를 최종 피처로 채택")
    if rmse_m3 < rmse_m2:
        print("      (해빙 면적 추가 시 성능이 하락했으나, 변동성을 함께 고려하면 하락폭이 다소 방어됨을 확인)")
    best_features = features_m1
elif best_rmse == rmse_m2:
    print("결론 : 해빙 면적 변수를 포함한 모델이 가장 우수하여 이를 최종 피처로 채택")
    best_features = features_m2
else:
    print("결론 : 해빙 면적과 변동성(Std) 변수를 모두 포함한 모델이 가장 우수하여 이를 최종 피처로 채택")
    best_features = features_m3

# 5. Optuna 기반 LightGBM 하이퍼파라미터 튜닝
print("\n5. Optuna Hyperparameter Tuning")

# 선택된 피처셋을 대상으로 최적의 하이퍼파라미터 탐색
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'verbose': -1
    }

    cv_rmses = []
    X, y = train_df[best_features], train_df[TARGET]

    for train_idx, val_idx in tscv.split(X):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        cv_rmses.append(np.sqrt(mean_squared_error(y_val, preds)))

    return np.mean(cv_rmses)

# Optuna를 이용한 하이퍼파라미터 탐색
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='minimize', sampler=sampler)
study.optimize(objective, n_trials=500)
best_lgb_params = study.best_params
print(f" 최적 파라미터: {best_lgb_params}")

# 6. 모델 성능 비교 및 최종 모델 선정
print("\n6. 모델 비교 (2024~2025 검증)")

# 학습 데이터와 테스트 데이터 구성
X_train, y_train = train_df[best_features], train_df[TARGET]
X_test, y_test = test_df[best_features], test_df[TARGET]

# LightGBM, XGBoost, CatBoost 모델 생성
models = {
    'LightGBM (Tuned)': lgb.LGBMRegressor(**best_lgb_params, random_state=42, verbose=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=300, random_state=42, learning_rate=0.05, max_depth=5, objective='reg:squarederror'),
    'CatBoost': CatBoostRegressor(iterations=300, random_state=42, learning_rate=0.05, depth=5, verbose=0)
}

# 각 모델의 성능 평가
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    predictions[name] = preds
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f" {name:16s} Test RMSE: {rmse:.4f}")

# 앙상블 모델 성능 확인
# Ensemble은 최고 성능을 보였지만, 개별 변수 영향 해석(SHAP)의 일관성을 위해 단일 모델인 LightGBM을 최종 모델로 선정
preds_ens = (predictions['LightGBM (Tuned)'] + predictions['XGBoost'] + predictions['CatBoost']) / 3
ens_rmse = np.sqrt(mean_squared_error(y_test, preds_ens))
print(f" {'Ensemble':16s} Test RMSE: {ens_rmse:.4f}")

print("\n단일 LightGBM 모델을 최종 모델로 선정")
final_model = models['LightGBM (Tuned)']

# 7. 2026년 하반기 기온 예측
print("\n7. [최종 예측 결과] 2026년 7월~12월 유럽 평균 기온")

# 전체 데이터를 이용하여 최종 모델 재학습
final_model.fit(df[best_features], df[TARGET])

# 최근 5년(2021~2025) 평균 패턴을 이용해 입력 데이터 생성
recent_years_df = df[df['year'] >= 2021]
recent_monthly_avg = recent_years_df.groupby('month')[best_features].mean()

# 2026년 7~12월 기온 예측
for m in range(7, 13):
    pred_row = recent_monthly_avg.loc[m].copy()
    if 'year' in best_features:
        pred_row['year'] = 2026
    pred_row['month_sin'] = np.sin(2 * np.pi * m / 12)
    pred_row['month_cos'] = np.cos(2 * np.pi * m / 12)

    pred_df = pd.DataFrame([pred_row], columns=best_features)
    pred_val = final_model.predict(pred_df)[0]

    print(f" 2026년 {m:02d}월 예측 기온: {pred_val:6.3f} °C")

4. Ablation Study (TimeSeriesSplit)
Model                | RMSE     | ΔRMSE    | Improvement 
-------------------------------------------------------
No Sea Ice           | 0.9194   | -        | -           
+ Sea Ice            | 0.9425   | +0.0230  | -2.51%
+ Sea Ice Std        | 0.9248   | -0.0176  | +1.87%
-------------------------------------------------------
결론 : 해빙 변수를 제외한 모델(No Sea Ice)이 가장 우수하여 이를 최종 피처로 채택
      (해빙 면적 추가 시 성능이 하락했으나, 변동성을 함께 고려하면 하락폭이 다소 방어됨을 확인)

5. Optuna Hyperparameter Tuning
 최적 파라미터: {'n_estimators': 118, 'learning_rate': 0.08322122832198935, 'max_depth': 4, 'num_leaves': 39, 'subsample': 0.8453038902445653, 'colsample_bytree': 0.6241341816332271}

6. 모델 비교 (2024~2025 검증)
 LightGBM (Tuned) Test RMSE: 0.6091
 XGBoost          Test RMSE: 0.6509
 CatBoost         Test RMSE: 0.6795
 Ensemble         Test RMSE: 0.6106

단일 LightGBM 모델을 최종 모델로 선정

7. [최종 예측 결과] 2026년 7월~12월 유럽 평균 기온
 2026년 07월 예측 기온: 17.819 °C
 2026년 08월 예측 기온: 17.566 °C
 2026년 09월 예측 기온: 14.